# Notebook C — PeMS Bay Area: Data, EDA, SQL and handover

**PeMS-specific RAM-safe adaptation of the common Notebook C template.**

This notebook keeps the required structure of the common template—source registry, source comparison, cleaning log, EDA, SQL/RQ1, leakage checks, tests, handover, Audit Log, exercises and checklist—but maps it to the group's actual project:

- **Primary data:** Caltrans PeMS District 4, 100 stations on US-101, I-280 and I-880, 2023–2024, already standardized to 15-minute data in `data/processed/pems_d4.parquet` by Notebook A.
- **Secondary data:** NOAA/NCEI San Jose hourly weather in `data/raw/san_jose_hourly.csv`.
- **Target:** `speed`.
- **Unit:** `station`.
- **Frequency:** 15 minutes.
- **Forecast horizons:** 15, 30 and 60 minutes (`1, 2, 4` steps).
- **Daily seasonal lag:** 96 steps.

### Important handoff rule
Notebook A is the authoritative data-preparation pipeline. This notebook **audits and validates** that handoff instead of re-reading the 20+ GB raw PeMS files or silently changing the already-cleaned data. It does not overwrite an existing `feat.parquet`/`manifest.json` unless the handoff is missing.

In [1]:
# =========================================================
# CONFIG: PeMS Bay Area project
# =========================================================

TOPIC = (
    "Short-term Speed Forecasting on Bay Area Technology Corridors "
    "from Caltrans PeMS with Weather and Office-day Covariates"
)
GROUP = "ADY201m"

PRIMARY_SOURCE = {
    "name": "Caltrans PeMS District 4 — processed 15-minute Bay Area data",
    "url": "https://pems.dot.ca.gov/",
    "license": "Caltrans PeMS data; use subject to PeMS data-use terms",
    "path": "data/processed/pems_d4.parquet",
    "note": "Handoff produced/validated by Notebook A from 2023–2024 PeMS D4 data."
}

SECOND_SOURCE = {
    "name": "NOAA/NCEI Integrated Surface Database — San Jose hourly weather",
    "url": "https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database",
    "license": "NOAA/NCEI public data",
    "path": "data/raw/san_jose_hourly.csv",
    "note": "Hourly weather covariates joined by timestamp in Notebook A."
}

UNIT_COL = "station"
TIME_COL = "ts"
TARGET_COL = "speed"
EXOG_COLS = ["temp", "prcp"]
FREQ = "15min"
HORIZONS = [1, 2, 4]              # 15 / 30 / 60 minutes
SEASON = 96                       # 24 hours at 15-minute frequency
CONGESTION_THRESHOLD = 45.0       # mph, used for the PeMS congestion-share RQ1 query
TEST_DAYS = 92

# Keep evidence small enough for a 16 GB machine.
EDA_SAMPLE_N = 200_000
PROFILE_SAMPLE_N = 20_000
ACF_STATIONS_N = 10
CROSSCORR_STATIONS_N = 4

# Protect the existing Notebook-A handoff.
REBUILD_HANDOFF = False
WRITE_FULL_CLEAN_CSV = True
WRITE_FULL_PRIMARY_CSV = True
WRITE_FULL_FEAT_CSV = True
ANALYTIC_VIEW = "pems_clean"

print("Topic:", TOPIC)
print("Primary:", PRIMARY_SOURCE["path"])
print("Secondary:", SECOND_SOURCE["path"])

Topic: Short-term Speed Forecasting on Bay Area Technology Corridors from Caltrans PeMS with Weather and Office-day Covariates
Primary: data/processed/pems_d4.parquet
Secondary: data/raw/san_jose_hourly.csv


In [2]:
# =========================================================
# Shared imports and project helpers
# =========================================================

import os
import json
import glob
import gc
import hashlib
import datetime as dt
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, kruskal
from statsmodels.tsa.stattools import acf
from ydata_profiling import ProfileReport

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("sql").mkdir(parents=True, exist_ok=True)
Path("report").mkdir(parents=True, exist_ok=True)
Path("notebooks").mkdir(parents=True, exist_ok=True)

# Keep one DuckDB connection for all large-data queries.
con = duckdb.connect()

# RAM-safe settings for a 16 GB machine.
con.execute("SET memory_limit=\'10GB\'")
con.execute("SET threads=2")
con.execute("SET preserve_insertion_order=false")

TEAL = "#1B6B6D"
ACC = "#F4A261"
GREY = "#9FBFBF"
DARK = "#264653"


def style(ax):
    ax.spines[["top", "right"]].set_visible(False)


def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(f"report/{name}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved report/{name}.png")


def file_md5(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def safe_query_df(sql):
    """Use pandas only for compact query results, never for the full PeMS table."""
    return con.execute(sql).df()

print("pandas", pd.__version__, "| duckdb", duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


C:\Users\Winn\AppData\Local\Temp\ipykernel_28212\2948333666.py:22: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## Hướng dẫn hỏi AI (giữ yêu cầu của Notebook_C gốc)

1. Đọc phần hướng dẫn của bước đang làm trước.
2. Tự thử trước; khi cần AI, cung cấp dữ liệu/cột/code/lỗi thực tế.
3. Kiểm tra một lần bằng dữ liệu hoặc kết quả chạy thật trước khi dùng.
4. Chỉ ghi vào Audit Log những prompt thực sự đã thay đổi công việc; không tạo log giả.


## AI Audit Log helper

The common template requires prompts that changed the work to be logged. Keep only **real** prompts you actually used; do not run made-up examples as if they were real work.

In [3]:
# AI Audit Log helper
AUDIT_PATH = "report/ai_audit_log.csv"


def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    row = {
        "date": dt.date.today().isoformat(),
        "group": GROUP,
        "step": step,
        "prompt": prompt[:500],
        "tool": tool,
        "ai_output": ai_output_summary[:500],
        "verified_how": verified_how[:300],
        "decision": decision[:300],
        "hallucination": int(hallucination),
    }
    pd.DataFrame([row]).to_csv(
        AUDIT_PATH,
        mode="a",
        header=not os.path.exists(AUDIT_PATH),
        index=False,
    )
    print("audit entry saved:", step)

## Bước 0 — Môi trường và cấu trúc thư mục

**Giữ yêu cầu gốc:** cùng môi trường, có `requirements.txt`, thư mục chuẩn, và file báo cáo.

**Biến đổi cho PeMS:** không đọc raw 20+ GB trực tiếp vào Pandas. Mọi truy vấn lớn dùng DuckDB/Parquet; chỉ đưa sample hoặc aggregate nhỏ vào Pandas để vẽ/kiểm định.

In [4]:
# Step 0a: environment notes (run once in a terminal, not as a Python cell)
# Windows example:
#   .venv\Scripts\activate
#   pip install pandas numpy duckdb pyarrow scipy scikit-learn lightgbm xgboost \
#       statsmodels matplotlib seaborn ydata-profiling mapie shap jupyter
#   pip freeze > requirements.txt

for d in ["data/raw", "data/processed", "sql", "report", "notebooks"]:
    os.makedirs(d, exist_ok=True)

print("folders ready")

folders ready


## Bước 1 — Nguồn dữ liệu, phạm vi và kế hoạch thu thập

The common template requires two registered sources and a reproducible data statement. For this project, the raw acquisition was completed before this notebook: Notebook A produced `pems_d4.parquet` and merged hourly weather. Notebook C therefore validates the two-source handoff instead of re-downloading the huge raw PeMS files.

In [5]:
# Step 1a: source registry
sources = [
    {
        **PRIMARY_SOURCE,
        "downloaded": pd.Timestamp.today().date().isoformat(),
    },
    {
        **SECOND_SOURCE,
        "downloaded": pd.Timestamp.today().date().isoformat(),
    },
]

# Fill exact size/time metadata from the files without materializing the PeMS table.
pems_meta = safe_query_df(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT {UNIT_COL}) AS units,
        MIN({TIME_COL}) AS start,
        MAX({TIME_COL}) AS end,
        COUNT(*) FILTER (WHERE {TARGET_COL} IS NULL) AS missing_target
    FROM read_parquet('{PRIMARY_SOURCE["path"]}')
""").iloc[0]

weather_meta = safe_query_df(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(ts) AS start,
        MAX(ts) AS end
    FROM read_csv_auto('{SECOND_SOURCE["path"]}', header=true)
""").iloc[0]

registry = pd.DataFrame([
    {
        **PRIMARY_SOURCE,
        "downloaded": pd.Timestamp.today().date().isoformat(),
        "rows": int(pems_meta["rows"]),
        "units": int(pems_meta["units"]),
        "cols": len(safe_query_df(f"DESCRIBE SELECT * FROM read_parquet('{PRIMARY_SOURCE['path']}')")),
        "start": str(pems_meta["start"]),
        "end": str(pems_meta["end"]),
        "note": PRIMARY_SOURCE["note"],
    },
    {
        **SECOND_SOURCE,
        "downloaded": pd.Timestamp.today().date().isoformat(),
        "rows": int(weather_meta["rows"]),
        "units": np.nan,
        "cols": len(safe_query_df(f"DESCRIBE SELECT * FROM read_csv_auto('{SECOND_SOURCE['path']}', header=true)")),
        "start": str(weather_meta["start"]),
        "end": str(weather_meta["end"]),
        "note": SECOND_SOURCE["note"],
    },
])

registry.to_csv("report/data_sources.csv", index=False)
print("saved report/data_sources.csv")
display(registry)

saved report/data_sources.csv


,name,url,license,path,note,downloaded,rows,units,cols,start,end
0,Caltrans PeMS District 4 — processed 15-minute...,https://pems.dot.ca.gov/,Caltrans PeMS data; use subject to PeMS data-u...,data/processed/pems_d4.parquet,Handoff produced/validated by Notebook A from ...,2026-09-20,7017600,100.0,10,2023-01-01 00:00:00,2024-12-31 23:45:00
1,NOAA/NCEI Integrated Surface Database — San Jo...,https://www.ncei.noaa.gov/products/land-based-...,NOAA/NCEI public data,data/raw/san_jose_hourly.csv,Hourly weather covariates joined by timestamp ...,2026-09-20,17544,NaN,3,2023-01-01 00:00:00,2024-12-31 23:00:00


In [6]:
# Step 1b: register the two files as DuckDB views; no full-table pandas load
con.execute(f"CREATE OR REPLACE VIEW pems AS SELECT * FROM read_parquet('{PRIMARY_SOURCE['path']}')")
con.execute(f"CREATE OR REPLACE VIEW weather AS SELECT * FROM read_csv_auto('{SECOND_SOURCE['path']}', header=true)")

pems_desc = safe_query_df("DESCRIBE pems")
weather_desc = safe_query_df("DESCRIBE weather")

print("PeMS columns:")
display(pems_desc)
print("Weather columns:")
display(weather_desc)

PeMS columns:


,column_name,column_type,null,key,default,extra
0,station,INTEGER,YES,None,None,None
1,ts,TIMESTAMP,YES,None,None,None
2,speed,DOUBLE,YES,None,None,None
3,total_flow,DOUBLE,YES,None,None,None
4,occupancy,DOUBLE,YES,None,None,None
5,Fwy,BIGINT,YES,None,None,None
6,Dir,VARCHAR,YES,None,None,None
7,Lanes,BIGINT,YES,None,None,None
8,temp,DOUBLE,YES,None,None,None
9,prcp,DOUBLE,YES,None,None,None


Weather columns:


,column_name,column_type,null,key,default,extra
0,ts,TIMESTAMP,YES,None,None,None
1,temp,DOUBLE,YES,None,None,None
2,prcp,DOUBLE,YES,None,None,None


In [7]:
# Step 1c: source comparison — temporal overlap, coverage and weather agreement
comparison = safe_query_df(f"""
WITH primary_hours AS (
    SELECT
        date_trunc('hour', {TIME_COL}) AS ts_h,
        AVG(temp) AS temp_pems,
        AVG(prcp) AS prcp_pems
    FROM pems
    GROUP BY 1
),
weather_clean AS (
    SELECT
        CAST(ts AS TIMESTAMP) AS ts_h,
        AVG(temp) AS temp_weather,
        AVG(prcp) AS prcp_weather
    FROM weather
    GROUP BY 1
),
joined AS (
    SELECT *
    FROM primary_hours p
    LEFT JOIN weather_clean w USING (ts_h)
)
SELECT
    (SELECT MIN(ts) FROM pems) AS pems_start,
    (SELECT MAX(ts) FROM pems) AS pems_end,
    (SELECT MIN(ts) FROM weather) AS weather_start,
    (SELECT MAX(ts) FROM weather) AS weather_end,
    (SELECT COUNT(*) FROM primary_hours) AS pems_hours,
    (SELECT COUNT(*) FROM weather_clean) AS weather_hours,
    COUNT(*) FILTER (WHERE temp_weather IS NOT NULL) AS matched_hours,
    COUNT(*) FILTER (WHERE temp_weather IS NULL) AS pems_only_hours,
    (SELECT COUNT(*)
     FROM weather_clean w
     LEFT JOIN primary_hours p USING (ts_h)
     WHERE p.ts_h IS NULL) AS weather_only_hours,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE temp_weather IS NOT NULL)
        / NULLIF(COUNT(*), 0), 3
    ) AS temp_match_pct,
    corr(temp_pems, temp_weather) AS temp_corr,
    AVG(temp_pems - temp_weather) AS mean_temp_diff,
    corr(prcp_pems, prcp_weather) AS prcp_corr
FROM joined
""")

comparison.to_csv("report/table_source_comparison.csv", index=False)
display(comparison.round(4))


,pems_start,pems_end,weather_start,weather_end,pems_hours,weather_hours,matched_hours,pems_only_hours,weather_only_hours,temp_match_pct,temp_corr,mean_temp_diff,prcp_corr
0,2023-01-01,2024-12-31 23:45:00,2023-01-01,2024-12-31 23:00:00,17544,17544,17513,31,0,99.823,1.0,0.0,1.0


In [8]:
# Step 1d: PeMS scope / coverage check
scope = safe_query_df(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT station) AS stations,
    COUNT(DISTINCT Fwy) AS freeways,
    MIN(ts) AS start,
    MAX(ts) AS end,
    COUNT(DISTINCT date_trunc('day', ts)) AS days
FROM pems
""")

display(scope)

assert int(scope.loc[0, "stations"]) == 100, "Expected the project handoff to contain 100 stations."
assert str(scope.loc[0, "start"]).startswith("2023-01-01"), "Unexpected start date."
assert str(scope.loc[0, "end"]).startswith("2024-12-31"), "Unexpected end date."
freeway_vals = [int(r[0]) for r in con.execute("SELECT DISTINCT Fwy FROM pems WHERE Fwy IS NOT NULL ORDER BY Fwy").fetchall()]
assert freeway_vals == [101, 280, 880], f"Unexpected freeway scope: {freeway_vals}"


,rows,stations,freeways,start,end,days
0,7017600,100,3,2023-01-01,2024-12-31 23:45:00,731


### CSV round-trip requirement, adapted for the large PeMS handoff

The common template asks for a CSV round-trip. Writing the entire 7M-row PeMS table to a second CSV is unnecessary duplication, so this PeMS version writes a **20,000-row deterministic audit snapshot** to `data/raw/primary_snapshot.csv` and checks its round-trip. The full data-of-record remains `pems_d4.parquet`.

In [9]:
# Step 1e: CSV round-trip of the full PeMS source without a full-table Pandas load
RAW_CSV = "data/raw/primary_snapshot.csv"

if WRITE_FULL_PRIMARY_CSV:
    con.execute(f"""
        COPY (
            SELECT *
            FROM pems
            ORDER BY station, ts
        )
        TO '{RAW_CSV}'
        (FORMAT CSV, HEADER TRUE)
    """)

    roundtrip_rows = int(
        con.execute(
            f"SELECT COUNT(*) FROM read_csv_auto('{RAW_CSV}', header=true)"
        ).fetchone()[0]
    )

    assert roundtrip_rows == int(pems_meta["rows"]),         "Full primary CSV round-trip row count mismatch."
    print("wrote full", RAW_CSV, "| rows", roundtrip_rows)
else:
    snapshot = safe_query_df(f"""
        SELECT *
        FROM pems
        ORDER BY station, ts
        LIMIT {PROFILE_SAMPLE_N}
    """)
    snapshot.to_csv(RAW_CSV, index=False)
    check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
    assert len(check) == len(snapshot)
    print("wrote audit snapshot", RAW_CSV, "| rows", len(check))
    del snapshot, check
    gc.collect()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote full data/raw/primary_snapshot.csv | rows 7017600


### Bước 1d: mở rộng thu thập dữ liệu (tùy chọn, giữ cấu trúc Notebook_C gốc)

Nguồn raw PeMS đã được thu thập trong Notebook A. Phần này chỉ dành cho trường hợp nhóm thật sự mở rộng thêm giai đoạn hoặc station; không chạy mặc định và không thay đổi handoff hiện tại.


In [10]:
# Step 1d: optional acquisition extension — disabled by default
EXTRA_PEMS_FILES = []  # add real files only when the scope is intentionally expanded

if EXTRA_PEMS_FILES:
    print("Additional files requested:", EXTRA_PEMS_FILES)
    print("Validate their schema, time range and station scope before changing Notebook A.")
else:
    print("No additional acquisition requested; Notebook A remains the data-acquisition source of record.")


No additional acquisition requested; Notebook A remains the data-acquisition source of record.


## Bước 2 — Source comparison and cleaning audit

The common template requires a cleaning log. Because Notebook A has already created the authoritative cleaned 15-minute PeMS handoff, this notebook **validates the cleaning conditions and records the resulting counts without changing the handoff**. This avoids a second, potentially different cleaning pipeline.

In [11]:
# Step 2a: cleaning audit and project-specific clean view
# Notebook A remains authoritative. This step does NOT overwrite pems_d4.parquet.
# It records the target-cleaning rule required by the common template and
# exposes a derived DuckDB view used for C's RQ1/SQL/EDA work.

clean_metrics = safe_query_df("""
WITH dup AS (
    SELECT station, ts, COUNT(*) AS c
    FROM pems
    GROUP BY station, ts
    HAVING COUNT(*) > 1
),
summary AS (
    SELECT
        COUNT(*) AS rows,
        COUNT(*) FILTER (WHERE ts IS NULL) AS bad_time,
        COUNT(*) FILTER (WHERE speed IS NULL) AS missing_speed,
        COUNT(*) FILTER (
            WHERE speed IS NOT NULL
              AND NOT (speed > 0 AND speed < 100)
        ) AS invalid_speed,
        COUNT(DISTINCT station) AS stations
    FROM pems
)
SELECT
    rows,
    bad_time,
    missing_speed,
    invalid_speed,
    (SELECT COALESCE(SUM(c - 1), 0) FROM dup) AS duplicate_excess,
    stations
FROM summary
""").iloc[0]

clean_log = pd.DataFrame([
    ["parse time", int(clean_metrics.bad_time), 0,
     "processed parquet already has typed timestamps"],
    ["drop duplicates", int(clean_metrics.duplicate_excess), 0,
     "station-timestamp uniqueness audit"],
    ["drop missing target", int(clean_metrics.missing_speed),
     int(clean_metrics.missing_speed),
     "speed is the forecast target; excluded only in the derived clean view"],
    ["physical speed range", int(clean_metrics.invalid_speed),
     int(clean_metrics.invalid_speed),
     "expected 0 < speed < 100 mph"],
], columns=["check", "rows_flagged", "rows_excluded_in_clean_view", "reason"])

clean_log.to_csv("report/table_cleaning_log.csv", index=False)
display(clean_log)

assert int(clean_metrics.bad_time) == 0
assert int(clean_metrics.duplicate_excess) == 0
assert int(clean_metrics.invalid_speed) == 0
assert int(clean_metrics.stations) == 100

con.execute("""
CREATE OR REPLACE VIEW pems_clean AS
SELECT *
FROM pems
WHERE ts IS NOT NULL
  AND speed IS NOT NULL
  AND speed > 0
  AND speed < 100
""")

clean_meta = safe_query_df("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT station) AS stations,
       MIN(ts) AS start,
       MAX(ts) AS end,
       COUNT(*) FILTER (WHERE speed IS NULL) AS missing_speed
FROM pems_clean
""").iloc[0]

print("Clean analysis view:", dict(clean_meta))
assert int(clean_meta["missing_speed"]) == 0
assert int(clean_meta["stations"]) == 100


,check,rows_flagged,rows_excluded_in_clean_view,reason
0,parse time,0,0,processed parquet already has typed timestamps
1,drop duplicates,0,0,station-timestamp uniqueness audit
2,drop missing target,4500,4500,speed is the forecast target; excluded only in...
3,physical speed range,0,0,expected 0 < speed < 100 mph


Clean analysis view: {'rows': np.int64(7013100), 'stations': np.int64(100), 'start': Timestamp('2023-01-01 00:00:00'), 'end': Timestamp('2024-12-31 23:45:00'), 'missing_speed': np.int64(0)}


In [12]:
# Step 2b: regular-grid/coverage audit
# Target-missing timestamps have been excluded by the derived clean view.
# Report the resulting grid gaps; do not synthesize speed values in C.
coverage = safe_query_df("""
SELECT
    station,
    MIN(ts) AS start_ts,
    MAX(ts) AS end_ts,
    COUNT(*) AS observed_rows,
    CAST(date_diff('minute', MIN(ts), MAX(ts)) / 15 AS BIGINT) + 1 AS expected_rows,
    (CAST(date_diff('minute', MIN(ts), MAX(ts)) / 15 AS BIGINT) + 1) - COUNT(*) AS missing_grid_rows
FROM pems_clean
GROUP BY station
ORDER BY station
""")

coverage.to_csv("report/table_station_coverage.csv", index=False)
print(
    "stations with missing grid rows after target cleaning:",
    int((coverage.missing_grid_rows > 0).sum())
)
display(coverage.head())


stations with missing grid rows after target cleaning: 100


,station,start_ts,end_ts,observed_rows,expected_rows,missing_grid_rows
0,400000,2023-01-01,2024-12-31 23:45:00,70131,70176,45
1,400001,2023-01-01,2024-12-31 23:45:00,70131,70176,45
2,400002,2023-01-01,2024-12-31 23:45:00,70131,70176,45
3,400006,2023-01-01,2024-12-31 23:45:00,70131,70176,45
4,400007,2023-01-01,2024-12-31 23:45:00,70131,70176,45


In [13]:
# Step 2c: weather-match and target-missingness audit
weather_match = safe_query_df("""
SELECT
    COUNT(*) AS rows,
    AVG(CASE WHEN temp IS NOT NULL THEN 1.0 ELSE 0.0 END) AS temp_nonmissing_rate,
    AVG(CASE WHEN prcp IS NOT NULL THEN 1.0 ELSE 0.0 END) AS prcp_nonmissing_rate
FROM pems_clean
""")

weather_match.to_csv("report/table_weather_match.csv", index=False)
display(weather_match.round(4))

target_missing = safe_query_df("""
SELECT
    COUNT(*) AS source_rows,
    COUNT(*) FILTER (WHERE speed IS NULL) AS source_missing_speed
FROM pems
""")

target_missing.to_csv("report/table_target_missing_audit.csv", index=False)
display(target_missing)


,rows,temp_nonmissing_rate,prcp_nonmissing_rate
0,7013100,0.9982,0.9591


,source_rows,source_missing_speed
0,7017600,4500


In [14]:
# Step 2d: write the derived clean audit table; never overwrite Notebook-A feat/manifest.
if WRITE_FULL_CLEAN_CSV:
    con.execute("""
        COPY (SELECT * FROM pems_clean ORDER BY station, ts)
        TO 'data/processed/clean.csv'
        (FORMAT CSV, HEADER TRUE)
    """)
    print("wrote data/processed/clean.csv directly from DuckDB")

con.execute("""
    COPY (SELECT * FROM pems_clean ORDER BY station, ts)
    TO 'data/processed/clean.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

clean_file_meta = safe_query_df("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT station) AS stations,
       MIN(ts) AS start,
       MAX(ts) AS end
FROM pems_clean
""").iloc[0]

print("wrote data/processed/clean.parquet")
print(
    f"Data statement: {PRIMARY_SOURCE['name']} + {SECOND_SOURCE['name']}; "
    f"{int(clean_file_meta['rows']):,} cleaned rows, "
    f"{int(clean_file_meta['stations'])} stations, "
    f"{clean_file_meta['start']} to {clean_file_meta['end']}, "
    f"15-minute working frequency."
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote data/processed/clean.csv directly from DuckDB
wrote data/processed/clean.parquet
Data statement: Caltrans PeMS District 4 — processed 15-minute Bay Area data + NOAA/NCEI Integrated Surface Database — San Jose hourly weather; 7,013,100 cleaned rows, 100 stations, 2023-01-01 00:00:00 to 2024-12-31 23:45:00, 15-minute working frequency.


## EDA (RQ1) — RAM-safe PeMS analysis

The common template asks for profile, target distribution, time series, seasonality, unit heterogeneity, covariate relationships, lag relationships, ACF and missingness/regime checks. The PeMS version keeps all nine analyses but uses **DuckDB aggregates and fixed-size samples** instead of loading all 7M rows into Pandas.

In [15]:
# EDA setup: fixed 200k-row sample, plus small full-data aggregates
eda = safe_query_df(f"""
SELECT *
FROM pems_clean
USING SAMPLE reservoir({EDA_SAMPLE_N} ROWS) REPEATABLE (42)
""")

eda[TIME_COL] = pd.to_datetime(eda[TIME_COL])
print("EDA sample:", eda.shape)

EDA sample: (200000, 10)


In [16]:
# EDA 1: profile on a 20k subset, plus compact descriptive table
profile_sample = eda.sample(min(PROFILE_SAMPLE_N, len(eda)), random_state=42)
ProfileReport(
    profile_sample,
    title=f"Profile {GROUP} — PeMS",
    minimal=True
).to_file("report/profile.html")

desc = eda[[TARGET_COL] + [c for c in EXOG_COLS if c in eda.columns]].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T.round(3)

desc.to_csv("report/table_describe.csv")
display(desc)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 810.13it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

,count,mean,std,min,1%,5%,50%,95%,99%,max
speed,200000.0,62.625,9.057,3.167,22.333,43.467,65.167,70.000,71.867,80.733
temp,199665.0,61.586,10.721,34.500,40.000,45.000,61.000,81.000,90.000,104.000
prcp,191958.0,0.002,0.012,0.000,0.000,0.000,0.000,0.001,0.050,0.280


In [17]:
# EDA 2: speed distribution and log-scale check
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(eda[TARGET_COL], bins=60, color=TEAL)
axes[0].set_xlabel("speed (mph)")
axes[0].set_ylabel("Count")
style(axes[0])

pos = eda[TARGET_COL][eda[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC)
axes[1].set_xlabel("log1p(speed)")
style(axes[1])

savefig(fig, "fig_eda_distribution")
print("speed skewness:", round(float(eda[TARGET_COL].skew()), 3))

saved report/fig_eda_distribution.png
speed skewness: -2.815


In [18]:
# EDA 3: time series of four representative stations
stations = safe_query_df("SELECT station FROM pems_clean GROUP BY station ORDER BY station LIMIT 4")["station"].tolist()
station_list = ", ".join(str(int(x)) for x in stations)

plot_df = safe_query_df(f"""
SELECT station, ts, speed
FROM pems_clean
WHERE station IN ({station_list})
ORDER BY station, ts
""")
plot_df["ts"] = pd.to_datetime(plot_df["ts"])

fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(stations, [TEAL, ACC, GREY, DARK]):
    g = plot_df[plot_df["station"] == u]
    ax.plot(g["ts"], g["speed"], lw=0.7, color=col, label=str(u))

cut = pd.Timestamp(pems_meta["end"]) - pd.Timedelta(days=TEST_DAYS)
ax.axvline(cut, color="k", ls="--", lw=1, label="92-day test start")
ax.set_ylabel("speed (mph)")
ax.legend(frameon=False, ncol=5)
style(ax)
savefig(fig, "fig_eda_timeseries")

del plot_df

saved report/fig_eda_timeseries.png


In [19]:
# EDA 4: calendar seasonality from the FULL dataset using DuckDB aggregates
season = safe_query_df("""
SELECT
    EXTRACT(HOUR FROM ts)::INTEGER AS hour,
    EXTRACT(DOW FROM ts)::INTEGER AS dow,
    EXTRACT(MONTH FROM ts)::INTEGER AS month,
    AVG(speed) AS mean_speed
FROM pems_clean
GROUP BY 1, 2, 3
""")

# Separate aggregations are easier to read and keep plotting compact.
by_hour = safe_query_df("SELECT EXTRACT(HOUR FROM ts)::INTEGER AS hour, AVG(speed) AS mean_speed FROM pems_clean GROUP BY 1 ORDER BY 1")
by_dow = safe_query_df("SELECT EXTRACT(DOW FROM ts)::INTEGER AS dow, AVG(speed) AS mean_speed FROM pems_clean GROUP BY 1 ORDER BY 1")
by_month = safe_query_df("SELECT EXTRACT(MONTH FROM ts)::INTEGER AS month, AVG(speed) AS mean_speed FROM pems_clean GROUP BY 1 ORDER BY 1")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, frame, x, label in [
    (axes[0], by_hour, "hour", "hour of day"),
    (axes[1], by_dow, "dow", "day of week"),
    (axes[2], by_month, "month", "month"),
]:
    ax.plot(frame[x], frame["mean_speed"], marker="o", color=TEAL)
    ax.set_xlabel(label)
    style(ax)
axes[0].set_ylabel("mean speed (mph)")
savefig(fig, "fig_eda_seasonality")

season_strength = {
    "hour": round(float(by_hour.mean_speed.std() / season.mean_speed.std()), 3),
    "dow": round(float(by_dow.mean_speed.std() / season.mean_speed.std()), 3),
    "month": round(float(by_month.mean_speed.std() / season.mean_speed.std()), 3),
}
print("seasonal strength:", season_strength)

saved report/fig_eda_seasonality.png
seasonal strength: {'hour': 0.868, 'dow': 0.349, 'month': 0.052}


In [20]:
# EDA 5: heterogeneity across stations from FULL data
per_unit = safe_query_df("""
SELECT station,
       AVG(speed) AS mean,
       STDDEV(speed) AS std,
       MIN(speed) AS min,
       MAX(speed) AS max,
       COUNT(*) AS count
FROM pems_clean
GROUP BY station
ORDER BY mean
""")

per_unit.to_csv("report/table_per_unit.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL)
ax.set_xlabel("stations sorted by mean speed")
ax.set_ylabel("mean speed (mph)")
style(ax)
savefig(fig, "fig_eda_units")

ratio = float(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9))
print("ratio of largest to smallest station mean:", round(ratio, 3))

saved report/fig_eda_units.png
ratio of largest to smallest station mean: 1.248


In [21]:
# EDA 6: relation with weather covariates on the fixed sample
rows = []
for c in [c for c in EXOG_COLS if c in eda.columns]:
    ok = eda[[c, TARGET_COL]].dropna()
    rho, p = spearmanr(ok[c].to_numpy(), ok[TARGET_COL].to_numpy())
    rows.append([c, round(float(rho), 3), p, len(ok)])

    b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"))[TARGET_COL].mean()
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.plot(range(len(b)), b.values, marker="o", color=TEAL)
    ax.set_xlabel(c + " (deciles)")
    ax.set_ylabel("mean speed (mph)")
    style(ax)
    savefig(fig, f"fig_eda_covariate_{c}")

corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"])
corr_tbl.to_csv("report/table_covariate_corr.csv", index=False)
display(corr_tbl)

saved report/fig_eda_covariate_temp.png
saved report/fig_eda_covariate_prcp.png


,covariate,spearman_rho,p_value,n
0,temp,-0.290,0.000000e+00,199665
1,prcp,-0.061,8.390175e-156,191958


In [22]:
# EDA 7: weather lead/lag correlation on four representative stations
lags = list(range(0, SEASON + 1, 12))  # 0, 3h, 6h, ..., 24h at 15-minute frequency

cc_df = safe_query_df(f"""
SELECT station, ts, speed, temp, prcp
FROM pems_clean
WHERE station IN ({station_list})
ORDER BY station, ts
""")
cc_df["ts"] = pd.to_datetime(cc_df["ts"])

fig, ax = plt.subplots(figsize=(7, 3.2))
for c in ["temp", "prcp"]:
    vals=[]
    for k in lags:
        corrs=[]
        for _, g in cc_df.groupby("station"):
            corrs.append(g["speed"].corr(g[c].shift(k)))
        vals.append(np.nanmean(corrs))
    ax.plot(lags, vals, marker="o", label=c)
    if np.isfinite(vals).any():
        j=int(np.nanargmax(np.abs(vals)))
        print(c, "best absolute lag:", lags[j], "corr:", round(float(vals[j]),3))
ax.axhline(0, color=GREY)
ax.set_xlabel("lag (15-minute steps; positive = weather earlier)")
ax.set_ylabel("correlation")
ax.legend(frameon=False)
style(ax)
savefig(fig, "fig_eda_crosscorr")

del cc_df

temp best absolute lag: 84 corr: -0.245
prcp best absolute lag: 0 corr: -0.027
saved report/fig_eda_crosscorr.png


In [23]:
# EDA 8: average ACF across a small set of stations, including the 1-day seasonal lag (96)
acf_stations = safe_query_df(f"SELECT station FROM pems_clean GROUP BY station ORDER BY station LIMIT {ACF_STATIONS_N}")["station"].tolist()
acf_station_sql = ", ".join(str(int(x)) for x in acf_stations)
acf_df = safe_query_df(f"""
SELECT station, ts, speed
FROM pems_clean
WHERE station IN ({acf_station_sql})
ORDER BY station, ts
""")

acfs=[]
for _, g in acf_df.groupby("station"):
    vals = g.sort_values("ts")["speed"].dropna().to_numpy()
    acfs.append(acf(vals, nlags=2*SEASON, fft=True))

acf_mean = np.mean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2))
markerline, stemlines, baseline = ax.stem(range(len(acf_mean)), acf_mean, basefmt=" ")
ax.set_xlabel("lag (15-minute steps)")
ax.set_ylabel("ACF")
style(ax)
savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(float(acf_mean[1]),3), "| at seasonal lag 96:", round(float(acf_mean[SEASON]),3))

del acf_df, acfs

saved report/fig_eda_acf.png
ACF at lag 1: 0.926 | at seasonal lag 96: 0.561


In [24]:
# EDA 9: missingness pattern for the weather covariates + regime check

weather_missing = safe_query_df("""
SELECT
    DATE_TRUNC('month', ts) AS month,

    AVG(
        CASE
            WHEN temp IS NULL THEN 1.0
            ELSE 0.0
        END
    ) AS temp_missing_share,

    AVG(
        CASE
            WHEN prcp IS NULL THEN 1.0
            ELSE 0.0
        END
    ) AS prcp_missing_share

FROM pems_clean
GROUP BY 1
ORDER BY 1
""")

fig, ax = plt.subplots(figsize=(10, 3.4))

ax.plot(
    weather_missing["month"],
    weather_missing["temp_missing_share"],
    label="temp",
    color=TEAL
)

ax.plot(
    weather_missing["month"],
    weather_missing["prcp_missing_share"],
    label="prcp",
    color=ACC
)

ax.set_xlabel("month")
ax.set_ylabel("missing share")
ax.legend(frameon=False)

style(ax)
savefig(fig, "fig_eda_missing")


roll = safe_query_df("""
SELECT
    DATE_TRUNC('month', ts) AS month,
    AVG(speed) AS mean_speed
FROM pems_clean
GROUP BY 1
ORDER BY 1
""")

cut = (
    pd.Timestamp(pems_meta["end"])
    - pd.Timedelta(days=TEST_DAYS)
)

fig, ax = plt.subplots(figsize=(10, 3.0))

ax.plot(
    roll["month"],
    roll["mean_speed"],
    color=TEAL,
    marker="o",
    ms=3
)

ax.axvline(
    cut,
    color="k",
    ls="--"
)

ax.set_ylabel("monthly mean speed (mph)")

style(ax)
savefig(fig, "fig_eda_regime")

print(
    "largest month-to-month change / mean:",
    round(
        float(
            roll["mean_speed"].diff().abs().max()
            / roll["mean_speed"].mean()
        ),
        3
    )
)

saved report/fig_eda_missing.png
saved report/fig_eda_regime.png
largest month-to-month change / mean: 0.017


### EDA evidence notes (required)

The five notes below must be replaced with statements based on the actual outputs before submission. Do not leave placeholders in the final report.

In [25]:
# EDA 10: evidence-backed findings generated from the actual EDA outputs.
season_min_row = roll.loc[roll["mean_speed"].idxmin()]
season_max_row = roll.loc[roll["mean_speed"].idxmax()]
season_min_month = pd.to_datetime(season_min_row["month"]).strftime("%Y-%m")
season_max_month = pd.to_datetime(season_max_row["month"]).strftime("%Y-%m")
season_min_speed = float(season_min_row["mean_speed"])
season_max_speed = float(season_max_row["mean_speed"])
month_change_ratio = float(roll["mean_speed"].diff().abs().max() / roll["mean_speed"].mean())

fastest_station = per_unit.loc[per_unit["mean"].idxmax()]
slowest_station = per_unit.loc[per_unit["mean"].idxmin()]
station_ratio = float(fastest_station["mean"] / max(slowest_station["mean"], 1e-9))

corr_valid = corr_tbl.dropna(subset=["spearman_rho"]).copy()
strongest_weather = corr_valid.iloc[corr_valid["spearman_rho"].abs().argmax()]
weather_name = strongest_weather["covariate"]
weather_rho = float(strongest_weather["spearman_rho"])
weather_p = float(strongest_weather["p_value"])

acf_lag1 = float(acf_mean[1])
acf_lag96 = float(acf_mean[SEASON])
source_row = comparison.iloc[0]
matched_pct = float(source_row["temp_match_pct"])
temp_corr = float(source_row["temp_corr"])
prcp_corr = float(source_row["prcp_corr"])

eda_notes = [
    f"1. Monthly mean speed ranged from {season_min_speed:.2f} mph in {season_min_month} to {season_max_speed:.2f} mph in {season_max_month}; the largest month-to-month change relative to the monthly mean was {month_change_ratio:.3f}.",
    f"2. Station mean speed ranged from {float(slowest_station['mean']):.2f} mph at station {int(slowest_station['station'])} to {float(fastest_station['mean']):.2f} mph at station {int(fastest_station['station'])}, a ratio of {station_ratio:.3f}.",
    f"3. The strongest absolute Spearman association among the weather covariates was {weather_name} (rho={weather_rho:.3f}, p={weather_p:.3g}) in the fixed EDA sample.",
    f"4. Average target ACF was {acf_lag1:.3f} at lag 1 (15 minutes) and {acf_lag96:.3f} at lag 96 (24 hours), quantifying short-term and daily dependence.",
    f"5. The hourly PeMS/weather comparison matched {matched_pct:.3f}% of PeMS hours; temperature correlation was {temp_corr:.3f} and precipitation correlation was {prcp_corr:.3f}."
]

assert len(eda_notes) == 5
assert all("INSERT" not in x for x in eda_notes)
open("report/eda_notes.md", "w", encoding="utf-8").write("\n".join(eda_notes))
print("saved report/eda_notes.md")
for note in eda_notes:
    print(note)


saved report/eda_notes.md
1. Monthly mean speed ranged from 62.09 mph in 2024-05 to 63.39 mph in 2023-04; the largest month-to-month change relative to the monthly mean was 0.017.
2. Station mean speed ranged from 54.17 mph at station 400284 to 67.60 mph at station 400319, a ratio of 1.248.
3. The strongest absolute Spearman association among the weather covariates was temp (rho=-0.290, p=0) in the fixed EDA sample.
4. Average target ACF was 0.926 at lag 1 (15 minutes) and 0.561 at lag 96 (24 hours), quantifying short-term and daily dependence.
5. The hourly PeMS/weather comparison matched 99.823% of PeMS hours; temperature correlation was 1.000 and precipitation correlation was 1.000.


## Bước 3 — DuckDB SQL for PeMS RQ1 and the exact A→B feature handoff

This section keeps the common notebook's SQL requirement but uses the **actual PeMS variables and forecast horizons**. The feature schema is kept compatible with Notebook B: `speed`, lagged speeds, 1-hour moving average, lagged flow, weather, calendar variables, and `y_15m/y_30m/y_60m`.

The SQL file is written as `sql/queries_C_PeMS.sql` and `sql/features_C_PeMS.sql` so this notebook does **not overwrite Notebook A's SQL files**.

In [26]:
# Step 3a: PeMS RQ1 SQL file — full data queried in DuckDB
SQL_RQ1 = r"""
-- Q1: speed profile by station, freeway, month and office-day grouping
SELECT
    station,
    Fwy,
    Dir,
    EXTRACT(MONTH FROM ts)::INTEGER AS month,
    CASE
        WHEN EXTRACT(DOW FROM ts)::INTEGER IN (2, 3, 4) THEN 'office_day'
        WHEN EXTRACT(DOW FROM ts)::INTEGER IN (1, 5) THEN 'other_weekday'
        ELSE 'weekend'
    END AS day_group,
    AVG(speed) AS mean_speed,
    STDDEV(speed) AS sd_speed,
    COUNT(*) AS n
FROM pems_clean
GROUP BY 1, 2, 3, 4, 5
ORDER BY 1, 2, 3, 4, 5;

-- Q2: morning congestion share by station and rain condition
SELECT
    station,
    (prcp > 0) AS rain,
    AVG((speed < 45.0)::INTEGER) AS congestion_share,
    COUNT(*) AS n
FROM pems_clean
WHERE EXTRACT(HOUR FROM ts)::INTEGER BETWEEN 7 AND 9
GROUP BY 1, 2
ORDER BY station, rain;

-- Q3: mean speed by precipitation decile
WITH dec AS (
    SELECT
        NTILE(10) OVER (ORDER BY prcp) AS prcp_decile,
        speed
    FROM pems_clean
    WHERE prcp IS NOT NULL
)
SELECT
    prcp_decile,
    AVG(speed) AS mean_speed,
    COUNT(*) AS n
FROM dec
GROUP BY 1
ORDER BY 1;

-- Q4: seasonal lag association (96 x 15 minutes = 1 day)
SELECT
    corr(speed, lag_speed) AS seasonal_lag_corr
FROM (
    SELECT
        speed,
        LAG(speed, 96) OVER (PARTITION BY station ORDER BY ts) AS lag_speed
    FROM pems_clean
) t;
"""

SQL_PATH = "sql/queries_C_PeMS.sql"
open(SQL_PATH, "w", encoding="utf-8").write(SQL_RQ1)
print(SQL_RQ1)



-- Q1: speed profile by station, freeway, month and office-day grouping
SELECT
    station,
    Fwy,
    Dir,
    EXTRACT(MONTH FROM ts)::INTEGER AS month,
    CASE
        WHEN EXTRACT(DOW FROM ts)::INTEGER IN (2, 3, 4) THEN 'office_day'
        WHEN EXTRACT(DOW FROM ts)::INTEGER IN (1, 5) THEN 'other_weekday'
        ELSE 'weekend'
    END AS day_group,
    AVG(speed) AS mean_speed,
    STDDEV(speed) AS sd_speed,
    COUNT(*) AS n
FROM pems_clean
GROUP BY 1, 2, 3, 4, 5
ORDER BY 1, 2, 3, 4, 5;

-- Q2: morning congestion share by station and rain condition
SELECT
    station,
    (prcp > 0) AS rain,
    AVG((speed < 45.0)::INTEGER) AS congestion_share,
    COUNT(*) AS n
FROM pems_clean
WHERE EXTRACT(HOUR FROM ts)::INTEGER BETWEEN 7 AND 9
GROUP BY 1, 2
ORDER BY station, rain;

-- Q3: mean speed by precipitation decile
WITH dec AS (
    SELECT
        NTILE(10) OVER (ORDER BY prcp) AS prcp_decile,
        speed
    FROM pems_clean
    WHERE prcp IS NOT NULL
)
SELECT
    prcp_decile,
   

In [27]:
# Step 3b: execute each RQ1 query and save evidence tables

def run_sql_text(sql_text, prefix="table_q"):
    body = "\n".join(
        line for line in sql_text.splitlines()
        if not line.strip().startswith("--")
    )
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt:
            continue
        k += 1
        out = con.execute(stmt).df()
        out.to_csv(f"report/{prefix}{k}.csv", index=False)
        print(f"Q{k}: {out.shape}")
        display(out.head(8))
    return k

run_sql_text(SQL_RQ1)

Q1: (3600, 8)


,station,Fwy,Dir,month,day_group,mean_speed,sd_speed,n
0,400000,101,S,1,office_day,64.521631,9.963229,2592
1,400000,101,S,1,other_weekday,66.180015,5.003073,1728
2,400000,101,S,1,weekend,67.351246,2.190026,1632
3,400000,101,S,2,office_day,64.317521,10.402583,2400
4,400000,101,S,2,other_weekday,66.705599,4.041866,1536
5,400000,101,S,2,weekend,67.505686,1.955723,1536
6,400000,101,S,3,office_day,63.984041,10.535778,2496
7,400000,101,S,3,other_weekday,66.236130,5.465102,1728


Q2: (300, 4)


,station,rain,congestion_share,n
0,400000,False,0.240674,7720
1,400000,True,0.163408,716
2,400000,<NA>,0.291667,336
3,400001,False,0.090674,7720
4,400001,True,0.029330,716
5,400001,<NA>,0.270833,336
6,400002,False,0.100518,7720
7,400002,True,0.131285,716


Q3: (10, 3)


,prcp_decile,mean_speed,n
0,1,62.472126,672630
1,2,63.229874,672630
2,3,62.394892,672630
3,4,63.678972,672630
4,5,61.837297,672630
5,6,62.386851,672630
6,7,64.190190,672630
7,8,62.138537,672630


Q4: (1, 1)


,seasonal_lag_corr
0,0.673432


4

In [28]:
# Step 3c: statistical tests for RQ1 on a fixed 200k-row sample to control RAM
stats_sample = eda[["station", "speed", "temp", "prcp"]].copy()

groups = [
    g["speed"].to_numpy()
    for _, g in stats_sample.groupby("station")
    if len(g) > 30
]

H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [[
    "Kruskal-Wallis speed across stations (fixed EDA sample)",
    round(float(H), 3) if np.isfinite(H) else np.nan,
    p
]]

for c in ["temp", "prcp"]:
    ok = stats_sample[[c, "speed"]].dropna()
    rho, pv = spearmanr(ok[c].to_numpy(), ok["speed"].to_numpy())
    tests.append([
        f"Spearman speed vs {c} (fixed EDA sample)",
        round(float(rho), 3),
        pv
    ])

tests = pd.DataFrame(
    tests,
    columns=["test", "statistic", "p_value"]
)
tests.to_csv("report/table_rq1_tests.csv", index=False)
display(tests)

del stats_sample, groups

,test,statistic,p_value
0,Kruskal-Wallis speed across stations (fixed ED...,29634.111,0.000000e+00
1,Spearman speed vs temp (fixed EDA sample),-0.290,0.000000e+00
2,Spearman speed vs prcp (fixed EDA sample),-0.061,8.390175e-156


In [29]:
# Step 3d: exact PeMS feature view compatible with Notebook B
FEATURE_SQL = r"""
CREATE OR REPLACE VIEW feat_candidate AS
SELECT
    station,
    Fwy,
    ts,
    speed,
    total_flow,
    occupancy,
    Lanes,
    temp,
    prcp,
    EXTRACT(HOUR FROM ts)::INTEGER AS hr,
    EXTRACT(DOW FROM ts)::INTEGER AS dow,
    LAG(speed, 1)   OVER w AS s_lag1,
    LAG(speed, 4)   OVER w AS s_lag4,
    LAG(speed, 96)  OVER w AS s_lag1d,
    LAG(speed, 672) OVER w AS s_lag1w,
    AVG(speed) OVER (
        w ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS s_ma1h,
    LAG(total_flow, 1) OVER w AS f_lag1,
    LEAD(speed, 1) OVER w AS y_15m,
    LEAD(speed, 2) OVER w AS y_30m,
    LEAD(speed, 4) OVER w AS y_60m
FROM pems_clean
WINDOW w AS (
    PARTITION BY station
    ORDER BY ts
);
"""

FEATURE_PATH = "sql/features_C_PeMS.sql"
open(FEATURE_PATH, "w", encoding="utf-8").write(FEATURE_SQL)
con.execute(FEATURE_SQL)

print("Feature view created: feat_candidate")

Feature view created: feat_candidate


In [30]:
# Step 3e: leakage and target-alignment checks without loading the feature table
feat_meta = safe_query_df("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT station) AS stations,
    MIN(ts) AS start,
    MAX(ts) AS end,
    COUNT(*) FILTER (WHERE y_60m IS NOT NULL) AS rows_with_y60
FROM feat_candidate
""").iloc[0]

print("feature rows:", int(feat_meta["rows"]))
print("stations:", int(feat_meta["stations"]))
print("rows with y_60m:", int(feat_meta["rows_with_y60"]))

# Deterministic row-level target alignment check on one station.
first_station = int(
    safe_query_df("SELECT MIN(station) AS station FROM feat_candidate").iloc[0]["station"]
)

small = safe_query_df(f"""
SELECT *
FROM feat_candidate
WHERE station = {first_station}
ORDER BY ts
LIMIT 1000
""")

raw_small = safe_query_df(f"""
SELECT ts, speed
FROM pems_clean
WHERE station = {first_station}
ORDER BY ts
LIMIT 1004
""")

expected = raw_small["speed"].shift(-4).iloc[:1000].to_numpy()
actual = small["y_60m"].to_numpy()
ok = np.allclose(actual[:996], expected[:996], equal_nan=True)

print("y_60m alignment check:", ok)
assert ok, "y_60m does not match speed shifted 4 rows ahead."

del small, raw_small


feature rows: 7013100
stations: 100
rows with y_60m: 7012700
y_60m alignment check: True


In [31]:
# Step 3f: RQ1 figures based on SQL outputs
q2 = pd.read_csv("report/table_q2.csv")
q3 = pd.read_csv("report/table_q3.csv")

# Aggregate Q2 to station-level congestion share for a compact paper figure.
q2_station = q2.groupby("station", as_index=False)["congestion_share"].mean().sort_values(
    "congestion_share", ascending=False
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))

axes[0].bar(
    range(len(q2_station)),
    q2_station["congestion_share"],
    color=TEAL
)
axes[0].set_xlabel("stations sorted by mean congestion share")
axes[0].set_ylabel("share with speed < 45 mph")
style(axes[0])

axes[1].plot(
    q3["prcp_decile"],
    q3["mean_speed"],
    marker="o",
    color=ACC
)
axes[1].set_xlabel("precipitation decile")
axes[1].set_ylabel("mean speed (mph)")
style(axes[1])

savefig(fig, "fig_rq1_sql")


saved report/fig_rq1_sql.png


## Kiểm tra, bàn giao và ghi Audit Log

The template requires duplicate checks, a non-missing cleaned target, >30k rows, temporal ordering, no future features, and a manifest with row count, columns, date range and MD5; Audit Log records only real AI-assisted changes. This PeMS version also checks the expected 100 stations and the exact feature names consumed by Notebook B.

In [32]:
# Test cases — all asserts must pass before handover
expected_feature_cols = [
    "station", "Fwy", "ts", "speed", "total_flow", "occupancy",
    "Lanes", "temp", "prcp", "hr", "dow", "s_lag1", "s_lag4",
    "s_lag1d", "s_lag1w", "s_ma1h", "f_lag1", "y_15m", "y_30m", "y_60m"
]

# Schema check
feat_cols = [r[0] for r in con.execute("DESCRIBE feat_candidate").fetchall()]
assert feat_cols == expected_feature_cols, f"Unexpected feature schema: {feat_cols}"

# Core row/station/target checks
assert int(feat_meta["stations"]) == 100
assert int(feat_meta["rows"]) >= 30000
assert int(safe_query_df("SELECT COUNT(*) FROM feat_candidate WHERE speed IS NULL").iloc[0,0]) == 0
assert int(safe_query_df("SELECT COUNT(*) FROM (SELECT station, ts, COUNT(*) c FROM feat_candidate GROUP BY station, ts HAVING COUNT(*) > 1)").iloc[0,0]) == 0

# No future target variable is included as an input feature by name.
X_like = [
    "speed", "s_lag1", "s_lag4", "s_lag1d", "s_lag1w", "s_ma1h",
    "total_flow", "f_lag1", "occupancy", "Lanes", "temp", "prcp", "hr", "dow"
]
assert not any(c.startswith("y_") for c in X_like)

print("Tests C: OK")

Tests C: OK


In [33]:
# Handover: protect an existing Notebook-A feat.parquet; create it only if missing.
feat_path = "data/processed/feat.parquet"
manifest_path = "data/processed/manifest.json"

candidate_count = int(safe_query_df("SELECT COUNT(*) FROM feat_candidate").iloc[0,0])
candidate_start = str(safe_query_df("SELECT MIN(ts) FROM feat_candidate").iloc[0,0])
candidate_end = str(safe_query_df("SELECT MAX(ts) FROM feat_candidate").iloc[0,0])

if not os.path.exists(feat_path):
    con.execute(f"""
        COPY (SELECT * FROM feat_candidate ORDER BY station, ts)
        TO '{feat_path}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print("Created missing handoff:", feat_path)
else:
    existing_meta = safe_query_df(f"""
        SELECT COUNT(*) AS rows, MIN(ts) AS start, MAX(ts) AS end
        FROM read_parquet('{feat_path}')
    """).iloc[0]
    assert int(existing_meta["rows"]) == candidate_count, "Existing feat.parquet row count differs from the C-generated candidate."
    print("Existing feat.parquet verified; not overwritten.")

# Validate manifest / create one only if missing.
md5 = file_md5(feat_path)

manifest = {
    "group": GROUP,
    "topic": TOPIC,
    "rows": candidate_count,
    "cols": expected_feature_cols,
    "units": 100,
    "start": candidate_start,
    "end": candidate_end,
    "freq": FREQ,
    "horizons": HORIZONS,
    "season": SEASON,
    "unit_col": UNIT_COL,
    "time_col": TIME_COL,
    "target_col": TARGET_COL,
    "exog_cols": EXOG_COLS,
    "test_days": TEST_DAYS,
    "md5": md5,
}

if os.path.exists(manifest_path):
    old = json.load(open(manifest_path, encoding="utf-8"))
    assert int(old["rows"]) == candidate_count, "Existing manifest row count does not match the feature handoff."
    assert old["md5"] == md5, "Existing manifest MD5 does not match feat.parquet."
    print("Existing manifest verified; not overwritten.")
else:
    json.dump(manifest, open(manifest_path, "w", encoding="utf-8"), indent=2)
    print("Created missing manifest.json")

display(pd.Series(manifest))
# Optional CSV copy for the common template; disabled by default for disk/time reasons.
if WRITE_FULL_FEAT_CSV:
    con.execute("""
        COPY (SELECT * FROM read_parquet('data/processed/feat.parquet') ORDER BY station, ts)
        TO 'data/processed/feat.csv'
        (FORMAT CSV, HEADER TRUE)
    """)
    print("wrote data/processed/feat.csv directly from DuckDB")


Existing feat.parquet verified; not overwritten.
Existing manifest verified; not overwritten.


group                                                   ADY201m
topic         Short-term Speed Forecasting on Bay Area Techn...
rows                                                    7013100
cols          [station, Fwy, ts, speed, total_flow, occupanc...
units                                                       100
start                                       2023-01-01 00:00:00
end                                         2024-12-31 23:45:00
freq                                                      15min
horizons                                              [1, 2, 4]
season                                                       96
unit_col                                                station
time_col                                                     ts
target_col                                                speed
exog_cols                                          [temp, prcp]
test_days                                                    92
md5                            6dba4fed8

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote data/processed/feat.csv directly from DuckDB


## Bài tập mở rộng cho PeMS (giữ yêu cầu template, nhưng đổi sang nghiệp vụ của đề)

Làm ít nhất 3 trong 5 và ghi kết quả vào `report/exercises_C.md`.

1. **Nguồn thứ ba:** thêm lịch ngày lễ hoặc sự kiện Bay Area, ghép theo thời gian và báo tỷ lệ match.
2. **So sánh tần suất:** so sánh 15 phút với 1 giờ và nhận xét seasonal strength.
3. **Hai truy vấn SQL:** top-5 stations theo tháng và congestion share theo tháng.
4. **STL:** phân rã chuỗi tốc độ của một station trên dữ liệu hourly để giảm RAM.
5. **Leakage demo:** tạo một feature sai bằng `LEAD(speed, 1)` rồi chứng minh MAE giảm bất thường; chỉ dùng cho thí nghiệm, không đưa vào `feat.parquet`.

In [34]:
# Exercise 2 starter: compare 15-minute vs 1-hour seasonal variation
hourly = safe_query_df("""
SELECT
    station,
    date_trunc('hour', ts) AS ts,
    AVG(speed) AS speed
FROM pems_clean
GROUP BY 1, 2
ORDER BY 1, 2
""")

hourly["hour"] = pd.to_datetime(hourly["ts"]).dt.hour
strength_hour = (
    hourly.groupby("hour")["speed"].mean().std()
    / hourly["speed"].std()
)
print("hourly weekday? seasonal strength proxy:", round(float(strength_hour), 3))

del hourly

hourly weekday? seasonal strength proxy: 0.477


In [35]:
# Exercise 3 starter: top-5 stations by month and monthly congestion share
exercise_sql = r"""
WITH monthly AS (
    SELECT
        EXTRACT(MONTH FROM ts)::INTEGER AS month,
        station,
        AVG(speed) AS mean_speed,
        AVG((speed < 45.0)::INTEGER) AS congestion_share
    FROM pems_clean
    GROUP BY 1, 2
), ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY month ORDER BY mean_speed DESC) AS rk
    FROM monthly
)
SELECT *
FROM ranked
WHERE rk <= 5
ORDER BY month, rk;
"""

display(con.execute(exercise_sql).df().head(20))

,month,station,mean_speed,congestion_share,rk
0,1,400001,67.825980,0.001680,1
1,1,400350,67.749686,0.002184,2
2,1,400319,67.572021,0.000000,3
3,1,400214,67.489606,0.000000,4
4,1,400093,66.876697,0.000000,5
5,2,400350,68.637958,0.002924,1
6,2,400319,68.017389,0.001279,2
7,2,400001,67.668110,0.001279,3
8,2,400214,67.626681,0.000548,4
9,2,400093,67.335695,0.000183,5


In [36]:
# Exercise 4 starter: STL on one station after hourly aggregation (RAM-safe)
from statsmodels.tsa.seasonal import STL

u0 = int(safe_query_df("SELECT MIN(station) FROM pems_clean").iloc[0,0])
s = safe_query_df(f"""
SELECT date_trunc('hour', ts) AS ts, AVG(speed) AS speed
FROM pems_clean
WHERE station = {u0}
GROUP BY 1
ORDER BY 1
""")

s["ts"] = pd.to_datetime(s["ts"])
s_series = s.set_index("ts")["speed"].asfreq("h").interpolate(limit=3).dropna()

if len(s_series) > 3 * 24:
    stl = STL(s_series, period=24, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for ax, comp, label in zip(
        axes,
        [stl.trend, stl.seasonal, stl.resid],
        ["trend", "seasonal", "residual"]
    ):
        ax.plot(comp, color=TEAL, lw=0.8)
        ax.set_ylabel(label)
        style(ax)
    savefig(fig, "fig_exercise_stl")
    print("residual variance share:", round(float(stl.resid.var() / s_series.var()), 3))


del s

saved report/fig_exercise_stl.png
residual variance share: 0.385


In [37]:
# Create the exercise-report template; fill only exercises actually completed.
exercise_report = """# Notebook C — Exercise Log

1. Third source: [fill with actual result]
2. Frequency comparison: [fill with actual result]
3. SQL extensions: [fill with actual result]
4. STL decomposition: [fill with actual result]
5. Intentional leakage demo: [fill with actual result]

Do not report an exercise as completed unless it was actually run.
"""
open("report/exercises_C.md", "w", encoding="utf-8").write(exercise_report)
print("created report/exercises_C.md template")


created report/exercises_C.md template


## Tự đánh giá Notebook C — PeMS

In [38]:
# Self-evaluation checklist: fill scores before submission.
self_eval = pd.DataFrame([
    ["Sources and scope are explicit", "report/data_sources.csv"],
    ["Source comparison has numeric evidence", "report/table_source_comparison.csv"],
    ["Cleaning audit exists", "report/table_cleaning_log.csv"],
    ["EDA has evidence-backed observations", "report/eda_notes.md"],
    ["SQL RQ1 evidence exists", "report/table_q1..q4.csv; report/table_rq1_tests.csv"],
    ["Feature leakage check passes", "Step 3e test"],
    ["Handover manifest is verified", "data/processed/manifest.json"],
    ["AI Audit Log is real (not fabricated)", "report/ai_audit_log.csv"],
], columns=["criterion", "evidence"])

display(self_eval)

,criterion,evidence
0,Sources and scope are explicit,report/data_sources.csv
1,Source comparison has numeric evidence,report/table_source_comparison.csv
2,Cleaning audit exists,report/table_cleaning_log.csv
3,EDA has evidence-backed observations,report/eda_notes.md
4,SQL RQ1 evidence exists,report/table_q1..q4.csv; report/table_rq1_test...
5,Feature leakage check passes,Step 3e test
6,Handover manifest is verified,data/processed/manifest.json
7,AI Audit Log is real (not fabricated),report/ai_audit_log.csv


## Lỗi thường gặp và cách sửa — PeMS

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `MemoryError` | full PeMS loaded into Pandas | keep DuckDB views; only sample/aggregate to Pandas |
| `FileNotFoundError` | wrong `data/processed/pems_d4.parquet` or weather path | verify paths in CONFIG |
| duplicate station-time | corrupted handoff | stop and fix Notebook A; do not deduplicate silently here |
| missing speed | target gaps in the source handoff | record in `table_cleaning_log.csv`; exclude only in the derived clean view without overwriting Notebook A |
| missing weather | source mismatch or timestamp issue | check `table_source_comparison.csv` |
| wrong feature schema | C and B disagree | keep the exact `expected_feature_cols` list |
| leakage | a feature uses future speed | only `LAG` for inputs; `LEAD` only for `y_15m/y_30m/y_60m` |
| overwriting B handoff | rerunning C after A | keep `REBUILD_HANDOFF=False` unless intentionally rebuilding |

## Checklist before handover

- [ ] `report/data_sources.csv` has PeMS + NOAA/NCEI source metadata.
- [ ] `report/table_source_comparison.csv` reports temporal overlap and weather agreement evidence.
- [ ] `report/table_cleaning_log.csv` records target exclusions, duplicate checks and physical-range checks.
- [ ] `data/raw/primary_snapshot.csv` completed the full DuckDB-streamed CSV round-trip (or was intentionally run as an audit snapshot).
- [ ] `data/processed/clean.parquet` and, when enabled, `data/processed/clean.csv` are created from the derived clean analysis view.
- [ ] EDA outputs `profile.html`, `table_describe.csv`, the nine RQ1 EDA analyses, and `report/eda_notes.md` with 5 numeric findings.
- [ ] `sql/queries_C_PeMS.sql` runs and writes `report/table_q1.csv` ... `table_q4.csv` for office-day, rain/congestion, precipitation-decile and seasonal-lag evidence.
- [ ] `report/table_rq1_tests.csv` contains the p-values/statistics.
- [ ] `sql/features_C_PeMS.sql` matches the exact Notebook-B feature schema.
- [ ] Leakage/target-alignment tests pass.
- [ ] Existing `feat.parquet` and `manifest.json` are verified and not overwritten accidentally.
- [ ] `report/exercises_C.md` records only exercises actually completed.
- [ ] `report/ai_audit_log.csv` contains only real prompts used by the group.

**RAM-safe adaptation note:** the common template's full-table Pandas loading is replaced by DuckDB views/aggregates and bounded samples; the project-specific RQ1 evidence, feature schema and A→B handoff remain unchanged.


## Mẫu viết Data và RQ1 cho bài báo

**Data.** We use Caltrans PeMS District 4 data for 100 stations on US-101, I-280 and I-880 from January 2023 through December 2024 at 15-minute resolution, merged with hourly San Jose weather covariates. The final handoff contains {rows} rows across {units} stations; the source-comparison table reports the temporal overlap and weather match rate.

**RQ1.** Speed varies across stations and calendar periods; the station heterogeneity, daily/weekly/monthly patterns, weather associations, seasonal-lag correlation and congestion share are reported in the EDA/SQL tables and figures. Replace every placeholder with a number from `report/` before submission.